## Resources

### Install Required Libraries

In [29]:
!pip install -q \
langchain \
langchain-community \
langchain-openai \
langchain-huggingface \
langchain-chroma \
faiss-cpu \
pypdf \
sentence-transformers

### Verify Installations

In [30]:
import langchain
import langchain_openai
import langchain_huggingface
import langchain_chroma

print("Everything installed successfully!")

Everything installed successfully!


### Load PDF Document

In [31]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/book.pdf")
documents = loader.load()

print(f"Number of pages: {len(documents)}")

Number of pages: 441


### Split Documents into Chunks

In [32]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks: {len(chunks)}")

Total chunks: 1757


### Inspect First Chunk

In [33]:
print(chunks[0])

page_content='Undergraduate Topics in Computer Science
A Beginners 
Guide to Python 3 
Programming
John Hunt' metadata={'producer': 'GPL Ghostscript 9.26', 'creator': 'Adobe InDesign 14.0 (Windows)', 'creationdate': "D:20200501191236Z00'00'", 'moddate': "D:20200501191236Z00'00'", 'author': '0014323', 'title': '482387_1_En_Print.indd', 'source': '/content/book.pdf', 'total_pages': 441, 'page': 0, 'page_label': 'C1'}


### Inspect Chunk Content

In [34]:
print(chunks[0].page_content)

Undergraduate Topics in Computer Science
A Beginners 
Guide to Python 3 
Programming
John Hunt


### Inspect Chunk Metadata

In [35]:
print(chunks[0].metadata)

{'producer': 'GPL Ghostscript 9.26', 'creator': 'Adobe InDesign 14.0 (Windows)', 'creationdate': "D:20200501191236Z00'00'", 'moddate': "D:20200501191236Z00'00'", 'author': '0014323', 'title': '482387_1_En_Print.indd', 'source': '/content/book.pdf', 'total_pages': 441, 'page': 0, 'page_label': 'C1'}


### Initialize Embedding Model

In [36]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Test Embedding Model

In [37]:
vector = embedding_model.embed_query("What is Retrieval Augmented Generation?")

print(type(vector))
print(len(vector))
print(vector[:10])

<class 'list'>
384
[-0.1085294559597969, -0.02714593894779682, -0.05479291081428528, 0.05426321178674698, -0.023731768131256104, 0.06446243077516556, 0.04553418979048729, -0.057844068855047226, 0.022253820672631264, -0.03078436851501465]


### Create Vector Store

In [38]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("✅ Vector database created successfully!")

✅ Vector database created successfully!


### Perform Similarity Search

In [39]:
query = "What is machine learning?"

results = vector_store.similarity_search(query, k=3)

print(f"Found {len(results)} relevant chunks.")

Found 3 relevant chunks.


### Display Search Results

In [40]:
for i, doc in enumerate(results):
    print(f"\n===== Result {i+1} =====")
    print(doc.page_content[:500])
    print("\nMetadata:", doc.metadata)


===== Result 1 =====
or machine learning or have a scienti ﬁc ﬂavor.
The aim of this book is to introduce Python to those with little or very little
programming knowledge, and then to take them through to become an experienced
Python developer.
As such the earlier parts of the book introduce fundamental concepts such as
what a variable is and how a for loop works. In contrast, the later chapters introduce
advanced concepts such as functional programming, object orientation, and
exception handling.

Metadata: {'producer': 'GPL Ghostscript 9.26', 'creator': 'Adobe InDesign 14.0 (Windows)', 'creationdate': "D:20200501191236Z00'00'", 'moddate': "D:20200501191236Z00'00'", 'author': '0014323', 'title': '482387_1_En_Print.indd', 'source': '/content/book.pdf', 'total_pages': 441, 'page': 6, 'page_label': 'vii'}

===== Result 2 =====
• Process. This symbol represents one or more operations (or programming
statements/expressions) that in some way apply behaviour or change the state of
the syste

### Configure Retriever

In [41]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

### Test Retriever

In [42]:
query = "What is Python?"

docs = retriever.invoke(query)

for doc in docs:
    print(doc.page_content[:300])
    print("-" * 50)

Chapter 1
Introduction
1.1 What Is Python?
Python is a general-purpose programming language in a similar vein to other
programming languages that you might have heard of such as C++, JavaScript or
Microsoft’s C# and Oracle ’s Java.
It has been around for some considerable time having been originally
--------------------------------------------------
Preface
There is currently huge interest in the Python programming language. This is driven
by several factors; its use in schools with the Raspberry Pi platform, its ability to be
used for DevOps scripts, its use in data science and machine learning and of course
the language itself.
There are many
--------------------------------------------------
As a language it has gained in interest over recent years, particularly within the
commercial world, with many people wanting to learn the language. This increased
interest in Python is driven by several different factors:
1. Its ﬂexibility and simplicity which makes it easy to learn.
2. Its use

### Define Context Retrieval Function

In [43]:
def retrieve_context(question):
    docs = retriever.invoke(question)

    context = "\n\n".join([doc.page_content for doc in docs])

    return context

### Test Context Retrieval

In [44]:
question = "What is object oriented programming?"

context = retrieve_context(question)

print(context)

17.6 Further Reading
If you want to explore some of the ideas presented in this chapter in more detail
here are sone online references:
•
https://en.wikipedia.org/wiki/Object-oriented_programming This is the wikipe-
dia entry for Object Oriented Programming and thus provides a quick reference
to much of the terminology and history of the subject and acts asa jumping off
point for other references.
•
https://dev.to/charanrajgolla/beginners-guide— object-oriented-programming

object-oriented system merely by reading the source code. Some Python environ-
ments (such as the PyCharm IDE) alleviate this problem, to some extent, through
the use of sophisticated browsers.
17.5 Where Is the Structure in an OO Program?
People new to object orientation may be confused because they have lost one of the
key elements that they use to help them understand and structure a software system:
the main program body. This is because the objects and the interactions between

the main program body. This is be

### Install Transformer Libraries

In [45]:
!pip install -q transformers accelerate sentencepiece

### Install Langchain Google GenAI

In [46]:
!pip install -q -U langchain-google-genai

### Set Google API Key

In [47]:
import os

os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY"

### Initialize Generative AI Model

In [48]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

print("✅ Gemini Loaded Successfully!")

✅ Gemini Loaded Successfully!


### Test LLM Invocation

In [49]:
response = llm.invoke("What is Python?")

print(response.content)

Python is a **high-level, interpreted, general-purpose programming language**.

Let's break down what that means and why it's so popular:

### Key Characteristics:

1.  **High-Level:** This means you don't have to worry about low-level details like managing computer memory directly. Python handles much of that for you, allowing you to focus on solving the problem at hand.
2.  **Interpreted:** Unlike compiled languages (like C++ or Java), Python code is executed line by line by an interpreter at runtime, rather than being fully translated into machine code before execution. This makes the development process faster and easier for debugging.
3.  **General-Purpose:** Python isn't designed for one specific task. It can be used for a wide variety of applications, from web development to scientific computing.
4.  **Readability and Simplicity:** Python's syntax is designed to be very clear and readable, often resembling natural English. It uses indentation to define code blocks, which enforce

### Define RAG Prompt Template

In [50]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the question ONLY using the provided context.

If the answer is not in the context, say:
"I don't know based on the provided document."

Context:
{context}

Question:
{question}
""")

### Build RAG Chain

In [51]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

### Test RAG Chain

In [52]:
response = chain.invoke("What is python")

print(response)

Python is a general-purpose programming language in a similar vein to other programming languages that you might have heard of such as C++, JavaScript or Microsoft’s C# and Oracle ’s Java.
